# Phase 9 — Nonequilibrium Thermodynamic Interpretation and Entropy-Production-Related Analysis

## Synthetic equilibrium vs nonequilibrium benchmark

Phase 9 begins with a stochastic system whose thermodynamic behavior is analytically known before any estimator is applied to the experimental MSC01 trajectories.

The benchmark is the two-dimensional rotational Ornstein–Uhlenbeck process

$$
dX = A X\,dt + \sqrt{2D}\,dW,
$$

with

$$
A = -kI + \omega R.
$$

The fixed baseline parameters are:

- \(k = 1\)
- \(D = 1\)
- \(dt = 0.1\)
- \(n_{\mathrm{steps}} = 100000\)
- seed \(= 2031\)

Two conditions are compared:

- equilibrium: \(\omega = 0\)
- nonequilibrium: \(\omega = 1\)

For both conditions, the stationary covariance is

$$
C = \frac{D}{k}I = I.
$$

Therefore, the two systems can have the same stationary spatial distribution even though their dynamics differ.

For the rotational nonequilibrium process, the analytical physical entropy-production rate is

$$
\sigma = \frac{2\omega^2}{k}.
$$

Thus the pre-specified analytical values are:

- equilibrium: \(\sigma = 0\)
- nonequilibrium: \(\sigma = 2\)

Before estimating entropy production from paths, this section tests two simpler properties:

1. whether both simulations reproduce the expected stationary covariance;
2. whether the nonequilibrium process displays the expected signed rotational probability current.

The rotational-current observable is

$$
c_t =
x_t y_{t+1}
-
y_t x_{t+1}.
$$

Its stationary expectation is

$$
E[c_t]
=
\frac{2D}{k}
e^{-k\,dt}
\sin(\omega\,dt).
$$

This rotational-current observable is **not** an entropy-production estimator.

In [1]:
import numpy as np
import pandas as pd

from _path import PROJECT_ROOT

from src.thermo import (
    analytic_epr_rotational_ou,
    analytic_mean_rotational_increment,
    rotational_increments,
    simulate_rotational_ou,
    stationary_covariance_isotropic,
)

print("Project root:", PROJECT_ROOT)
print("NumPy version:", np.__version__)
print("Phase 9 synthetic setup: READY")

Project root: C:\Users\Abolfazl.PH\Desktop\cell-irreversibility
NumPy version: 2.0.1
Phase 9 synthetic setup: READY


In [2]:
K = 1.0
DIFFUSION = 1.0
DT = 0.1
N_STEPS = 100_000
SYNTHETIC_SEED = 2031

OMEGA_EQUILIBRIUM = 0.0
OMEGA_NONEQUILIBRIUM = 1.0


stationary_covariance_theory = (
    stationary_covariance_isotropic(
        k=K,
        diffusion=DIFFUSION,
    )
)

sigma_equilibrium_theory = (
    analytic_epr_rotational_ou(
        k=K,
        omega=OMEGA_EQUILIBRIUM,
    )
)

sigma_nonequilibrium_theory = (
    analytic_epr_rotational_ou(
        k=K,
        omega=OMEGA_NONEQUILIBRIUM,
    )
)

current_equilibrium_theory = (
    analytic_mean_rotational_increment(
        k=K,
        omega=OMEGA_EQUILIBRIUM,
        diffusion=DIFFUSION,
        dt=DT,
    )
)

current_nonequilibrium_theory = (
    analytic_mean_rotational_increment(
        k=K,
        omega=OMEGA_NONEQUILIBRIUM,
        diffusion=DIFFUSION,
        dt=DT,
    )
)


print("Fixed synthetic parameters")
print("--------------------------")
print("k:", K)
print("D:", DIFFUSION)
print("dt:", DT)
print("n_steps:", N_STEPS)
print("seed:", SYNTHETIC_SEED)

print()
print("Theoretical stationary covariance:")
print(stationary_covariance_theory)

print()
print("Theoretical physical EPR")
print("equilibrium:", sigma_equilibrium_theory)
print(
    "nonequilibrium:",
    sigma_nonequilibrium_theory,
)

print()
print("Theoretical mean rotational increment")
print(
    "equilibrium:",
    current_equilibrium_theory,
)
print(
    "nonequilibrium:",
    current_nonequilibrium_theory,
)

Fixed synthetic parameters
--------------------------
k: 1.0
D: 1.0
dt: 0.1
n_steps: 100000
seed: 2031

Theoretical stationary covariance:
[[1. 0.]
 [0. 1.]]

Theoretical physical EPR
equilibrium: 0.0
nonequilibrium: 2.0

Theoretical mean rotational increment
equilibrium: 0.0
nonequilibrium: 0.18066602190484835


In [3]:
equilibrium_path = simulate_rotational_ou(
    n_steps=N_STEPS,
    k=K,
    omega=OMEGA_EQUILIBRIUM,
    diffusion=DIFFUSION,
    dt=DT,
    seed=SYNTHETIC_SEED,
)

nonequilibrium_path = simulate_rotational_ou(
    n_steps=N_STEPS,
    k=K,
    omega=OMEGA_NONEQUILIBRIUM,
    diffusion=DIFFUSION,
    dt=DT,
    seed=SYNTHETIC_SEED,
)


print(
    "Equilibrium path shape:",
    equilibrium_path.shape,
)

print(
    "Nonequilibrium path shape:",
    nonequilibrium_path.shape,
)

print(
    "All equilibrium values finite:",
    np.isfinite(equilibrium_path).all(),
)

print(
    "All nonequilibrium values finite:",
    np.isfinite(nonequilibrium_path).all(),
)

Equilibrium path shape: (100001, 2)
Nonequilibrium path shape: (100001, 2)
All equilibrium values finite: True
All nonequilibrium values finite: True


In [4]:
equilibrium_covariance_empirical = np.cov(
    equilibrium_path.T,
    ddof=1,
)

nonequilibrium_covariance_empirical = np.cov(
    nonequilibrium_path.T,
    ddof=1,
)


equilibrium_rotational = rotational_increments(
    equilibrium_path
)

nonequilibrium_rotational = rotational_increments(
    nonequilibrium_path
)


equilibrium_current_empirical = (
    equilibrium_rotational.mean()
)

nonequilibrium_current_empirical = (
    nonequilibrium_rotational.mean()
)


synthetic_summary = pd.DataFrame(
    {
        "condition": [
            "equilibrium",
            "nonequilibrium",
        ],
        "omega": [
            OMEGA_EQUILIBRIUM,
            OMEGA_NONEQUILIBRIUM,
        ],
        "theory_epr": [
            sigma_equilibrium_theory,
            sigma_nonequilibrium_theory,
        ],
        "theory_mean_rotational_increment": [
            current_equilibrium_theory,
            current_nonequilibrium_theory,
        ],
        "empirical_mean_rotational_increment": [
            equilibrium_current_empirical,
            nonequilibrium_current_empirical,
        ],
        "empirical_var_x": [
            equilibrium_covariance_empirical[0, 0],
            nonequilibrium_covariance_empirical[0, 0],
        ],
        "empirical_var_y": [
            equilibrium_covariance_empirical[1, 1],
            nonequilibrium_covariance_empirical[1, 1],
        ],
        "empirical_cov_xy": [
            equilibrium_covariance_empirical[0, 1],
            nonequilibrium_covariance_empirical[0, 1],
        ],
    }
)

synthetic_summary

,condition,omega,theory_epr,theory_mean_rotational_increment,empirical_mean_rotational_increment,empirical_var_x,empirical_var_y,empirical_cov_xy
0,equilibrium,0.0,0.0,0.000000,-0.00034,0.992851,0.983713,-0.000967
1,nonequilibrium,1.0,2.0,0.180666,0.17810,0.992347,0.984064,-0.001169


### Initial synthetic benchmark result

The exact-transition simulations reproduce the expected stationary spatial statistics in both conditions.

For the equilibrium process, the empirical covariance is close to the theoretical stationary covariance

$$
C = I,
$$

and the empirical mean signed rotational increment is approximately zero:

$$
\langle c_t\rangle_{\mathrm{emp}}
=
-0.00034.
$$

For the nonequilibrium rotational process, the empirical covariance remains close to the same stationary covariance, but the dynamics display a clear positive rotational current.

The analytical prediction is

$$
\langle c_t\rangle_{\mathrm{theory}}
=
0.180666,
$$

while the simulation gives

$$
\langle c_t\rangle_{\mathrm{emp}}
=
0.17810.
$$

Thus, the equilibrium and nonequilibrium systems can have nearly indistinguishable stationary spatial distributions while exhibiting different temporal probability currents.

This demonstrates why stationary density alone is insufficient to diagnose nonequilibrium dynamics.

The rotational-current observable used here is a dynamical diagnostic and is **not** itself an entropy-production estimate.

## Path-probability-ratio validation criterion

The next benchmark directly compares forward and time-reversed path probabilities for the exactly sampled rotational Ornstein–Uhlenbeck process.

Three rates will be kept distinct:

$$
\sigma_{\mathrm{continuous}}
=
\frac{2\omega^2}{k},
$$

the continuous-time physical entropy-production rate;

$$
\dot I_{\mathrm{sampled,theory}}
=
\frac{
4e^{-2k\,dt}\sin^2(\omega dt)
}{
(1-e^{-2k\,dt})dt
},
$$

the exact forward/reverse path-space irreversibility rate of the process observed only every \(dt\);

and

$$
\dot I_{\mathrm{path,empirical}}
=
\frac{
\log P[\Gamma]
-
\log P[\Gamma^R]
}{
N_{\mathrm{steps}}\,dt
},
$$

the empirical rate calculated from the simulated trajectory.

The validation criteria are fixed before inspecting the empirical path-ratio result.

For the nonequilibrium benchmark:

- the theoretical sampled path-space rate must be positive;
- it must be smaller than the continuous-time physical entropy-production rate at the finite sampling interval \(dt=0.1\);
- the empirical path-log-ratio rate must agree with the exact sampled-theory rate to within **5% relative error**.

For the equilibrium benchmark:

- the theoretical continuous and sampled rates are exactly zero;
- because detailed balance holds analytically, the empirical path-log-ratio rate should be zero up to floating-point numerical error;
- an absolute empirical rate not exceeding \(10^{-10}\) simulation-time\(^{-1}\) will be treated as numerically zero.

These criteria are numerical-validation criteria for the synthetic benchmark. They are not significance thresholds for the later MSC01 experimental analysis.

The 5% tolerance will not be changed after inspection of the empirical path-ratio result merely to obtain a passing validation.